# 🌍 Projekt 1 — Klasifikace satelitních snímků EuroSAT

Dataset: satelitní snímky (RGB verze)
Zdroj: https://www.kaggle.com/datasets/apollo2506/eurosat-dataset?select=EuroSAT

In [4]:
# 1. Připojení Google Disku (pokud běžíme v Google Colabu)
from pathlib import Path
import sys

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        print("Připojuji Google Disk...")
        drive.mount("/content/drive")
    else:
        print("Google Disk je již připojen.")
except ImportError:
    print("Běžíme lokálně (Google Drive mount není potřeba).")

Připojuji Google Disk...


MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

In [ ]:
# 2. Instalace torchinfo (pokud v prostředí chybí)
try:
    import torchinfo
except ImportError:
    !pip install -q torchinfo
    import torchinfo

In [ ]:
# 3. Importy knihoven a Device setup
import os
import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

from torchvision import datasets

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Univerzální volba zařízení: v Colabu GPU (CUDA), lokálně CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Aktivní device:", device)

In [ ]:
# 4. Automatické vyhledání cesty k datasetu EuroSAT
candidate_paths = [
    Path("EuroSAT"),                                   # Lokální projekt
    Path("/content/EuroSAT"),                          # Lokální SSD v Colabu
    Path("/content/drive/MyDrive/EuroSAT"),            # Kořen Google Disku
    Path("/content/drive/MyDrive/ML/EuroSAT"),         # Podsložka ML na Disku
    Path("/content/drive/MyDrive/school/ML/EuroSAT"),  # Podsložka school/ML na Disku
]

DATA_DIR = next((p for p in candidate_paths if p.exists()), None)

# Pokud nebyl nalezen ve standardních cestách, prohledáme Google Disk
if DATA_DIR is None and Path("/content/drive/MyDrive").exists():
    print("Hledám složku EuroSAT na Google Disku...")
    found = list(Path("/content/drive/MyDrive").glob("**/EuroSAT"))
    if found:
        DATA_DIR = found[0]

assert DATA_DIR is not None, (
    "Složka EuroSAT nebyla nalezena! Ujistěte se, že máte připojený Google Disk "
    "nebo zkontrolujte název složky na svém Google Disku."
)

print(f"Dataset EuroSAT úspěšně nalezen v: {DATA_DIR.resolve()}")

In [ ]:
# 5. Načtení rozdělení splitů z CSV a výpis statistik tříd
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "validation.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

total_samples = len(train_df) + len(val_df) + len(test_df)
print(f"Počty snímků:")
print(f"  • Train:      {len(train_df):5d} ({len(train_df)/total_samples:.1%})")
print(f"  • Validation: {len(val_df):5d} ({len(val_df)/total_samples:.1%})")
print(f"  • Test:       {len(test_df):5d} ({len(test_df)/total_samples:.1%})")
print(f"  • Celkem:     {total_samples:5d}")

print("\nZastoupení jednotlivých tříd v trénovací sadě:")
print(train_df["ClassName"].value_counts())

In [ ]:
# 6. Načtení ukázky / všech snímků do PyTorch tenzorů a statistiky pixelů
def load_split(df, data_dir=DATA_DIR):
    images = [np.array(Image.open(data_dir / fname)) for fname in df["Filename"]]
    # převod: (N, H, W, C) -> (N, C, H, W) normalizováno do [0.0, 1.0]
    X = torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2).float() / 255.0
    y = torch.tensor(df["Label"].values, dtype=torch.long)
    return X, y

print("Načítám snímky do tenzorů...")
X_train, y_train = load_split(train_df)
X_val, y_val     = load_split(val_df)
X_test, y_test   = load_split(test_df)

print(f"\nRozměry tenzorů:")
print(f"  • X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"  • X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"  • X_test:  {X_test.shape}  | y_test:  {y_test.shape}")
print(f"  • Rozsah hodnot pixelů: min={X_train.min().item():.2f}, max={X_train.max().item():.2f}")

# Průměr a směrodatná odchylka přes RGB kanály
mean_rgb = X_train.mean(dim=[0, 2, 3])
std_rgb  = X_train.std(dim=[0, 2, 3])
print(f"\nStatistika RGB kanálů v train sadě (pro budoucí normalizaci):")
print(f"  • Průměr (Mean):  R={mean_rgb[0]:.4f}, G={mean_rgb[1]:.4f}, B={mean_rgb[2]:.4f}")
print(f"  • Směrodatná odchylka (Std):   R={std_rgb[0]:.4f}, G={std_rgb[1]:.4f}, B={std_rgb[2]:.4f}")

In [ ]:
# 7. Vykreslení ukázkového snímku pro všech 10 tříd
classes = sorted(train_df["ClassName"].unique())
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for idx, cls_name in enumerate(classes):
    ax = axes[idx // 5, idx % 5]
    row = train_df[train_df["ClassName"] == cls_name].iloc[0]
    img = Image.open(DATA_DIR / row["Filename"])
    ax.imshow(img)
    ax.set_title(f"{cls_name}\n(Label {row['Label']})", fontsize=9)
    ax.axis("off")

plt.suptitle("Ukázky všech 10 tříd EuroSAT (64×64 px)", fontsize=12)
plt.tight_layout()
plt.show()